### Environment and Packages

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import pandas as pd
import numpy as np
import csv

from pathlib import Path
from collections import Counter
from enum import Enum

### Data Analysis

In [ ]:
try:
    # Works when running a .py file
    ROOT = Path(__file__).parent.resolve()
except NameError:
    # Works in notebooks / interactive
    ROOT = Path.cwd().resolve()

print(ROOT)
DATA_DIR = "Data Directory"

df = pd.read_csv(
    DATA_DIR, 
    sep="\t", 
    encoding="utf-8-sig", 
    quoting=csv.QUOTE_MINIMAL, 
    quotechar='"', 
    escapechar="\\", 
    header=0, 
    engine="python",)

In [ ]:
class Cols(str, Enum):
    Record_Number = "Record Number"
    Category = "Category"
    Title = "Title"
    Token = "Token"
    Tag = "Tag"
    Token_Tag_List = "Token_Tag_List"

In [ ]:
print(df.columns)
print(df.index)
print(df.dtypes)
print(df.shape)

print(df.head())
# print(df.tail())

In [ ]:
# Sort by Record Number in ascending order
df = df.sort_values(by=Cols.Record_Number, ascending=True)

# If want to reset the index after sorting
df = df.reset_index(drop=True)
print(df.head())

In [ ]:
df[Cols.Title] = df[Cols.Title].astype("string")
df[Cols.Token] = df[Cols.Token].astype("string")
df[Cols.Tag] = df[Cols.Tag].astype("string")
print(df.dtypes)

In [ ]:
print(len(df['Title'].unique()))
print(len(df['Tag'].unique()))
print(df['Tag'].value_counts())

In [ ]:
# Printing value counts description for each column
for col in df.columns:
    print(col, '\n', df[col].value_counts().describe(include=all),'\n')

In [ ]:
print(df['Title'][0])

In [ ]:
unique_chars = Counter("".join(df['Title'].dropna().unique()))
print(len(unique_chars))
print(unique_chars)
print(unique_chars.most_common(20))

In [ ]:
record = df[df[Cols.Record_Number]==1]
print(record[[Cols.Token, Cols.Tag]])
print(record[Cols.Title].values[0])
print(", ".join(record[Cols.Token]))

In [ ]:
max_words = df[Cols.Title].str.split().str.len().max()
print("Maximum token length:", max_words)

longest_titles = df.loc[
    df[Cols.Title].str.split().str.len() == max_words, Cols.Title
]
# print("\nLongest title(s):")
# print(longest_titles)

min_words = df[Cols.Title].str.split().str.len().min()
print("Mininmum token length:", min_words)

smallest_titles = df.loc[
    df[Cols.Title].str.split().str.len() == min_words, Cols.Title
]
# print("\nSmallest title(s):")
# print(smallest_titles)

In [ ]:
def collapse_token_runs(tokens, tags):
    """
    tokens: list[str]
    tags:   list[object] (may contain <NA>)
    Returns: list of (list_of_tokens, tag) groups
             e.g. [(['a','b','c'], 'model'), (['d'], 'make')]
    """
    groups = []
    cur_tokens = []
    cur_tag = None

    for tok, tag in zip(tokens, tags):
        if pd.isna(tok):
            print("NA token found!", tok, tokens)
            continue  # skip NA tokens (if any)
        if pd.isna(tag):
            # continuation of the current group
            if not cur_tokens:
                # edge case: a record starts with NA — start a group with tag=None
                cur_tag = None
            cur_tokens.append(tok)
        else:
            # new (non-NA) tag starts a new group
            if cur_tokens:
                groups.append((cur_tokens, cur_tag))
            cur_tokens = [tok]
            cur_tag = tag

    if cur_tokens:
        groups.append((cur_tokens, cur_tag))

    return groups

In [ ]:
def group_tokens_by_tag(df):
    return (
        df.groupby(Cols.Record_Number)
          .apply(lambda g: pd.Series({
              Cols.Title: g[Cols.Title].iloc[0],  # keep one title
              Cols.Category: g[Cols.Category].iloc[0],  # keep one category
              Cols.Token_Tag_List: collapse_token_runs(
                  g[Cols.Token].tolist(),
                  g[Cols.Tag].tolist()
              )
              # "Token_Tag_List": list(zip(g[Cols.Token], g[Cols.Tag]))  # pair tokens & tags
          }), include_groups=False)
          .reset_index()
    )

In [ ]:
token_tag_grouped = group_tokens_by_tag(df)
print(df.head(1)[Cols.Title].values[0])
print(token_tag_grouped.head(1)[Cols.Token_Tag_List].values[0])


##### Mapping Tokens -> Tag

In [ ]:
train_df = df.sample(frac=0.95, random_state=42)
test_df = df.drop(train_df.index)

In [ ]:
mapping = (
    train_df.groupby([Cols.Token, Cols.Category], dropna=False)[Cols.Tag]
      .agg(lambda x: x.value_counts(dropna=True).index[0] if x.notna().any() else np.nan)
      .to_dict()
)
print(len(mapping))

In [ ]:
true_cnt = 0
false_cnt = 0
for idx, row in test_df.iterrows():
    token = row[Cols.Token]
    category = row[Cols.Category]
    true_tag = row[Cols.Tag]

    pred_tag = mapping.get((token, category), np.nan)
    
    if pd.isna(true_tag) and pd.isna(pred_tag):
        match = True
    elif pd.isna(true_tag) or pd.isna(pred_tag):
        match = False
    else:
        match = (true_tag == pred_tag)

    if match:
        true_cnt = true_cnt+1
    else:
        false_cnt = false_cnt+1
    # print(f"Token: {token}, True Tag: {true_tag}, Predicted Tag: {pred_tag}, Match: {match}")

print(len(test_df))
print(f"Total Matches: {true_cnt}, Total Mismatches: {false_cnt}")

In [ ]:
actual_test = group_tokens_by_tag(df)
print(df.head(1)[Cols.Title].values[0])
print(actual_test.head(1)[Cols.Token_Tag_List].values[0])
token_match = 0
token_mismatch = 0
for idx, row in actual_test.iterrows():
    # if idx != 0:
    #     continue
    title = row[Cols.Title]
    category = row[Cols.Category]
    tokens = title.split()
    token_tag_list = row[Cols.Token_Tag_List]
    pred_tag = []
    for token in tokens:
        cur_tag = mapping.get((token, category), np.nan)
        if pd.isna(cur_tag):
            cur_tag = None
        pred_tag.append(cur_tag)
    
    final_token_tag = collapse_token_runs(tokens, pred_tag)
    #print(final_token_tag)
    for t in final_token_tag:
        #print(t, "Actual:", token_tag_list, "Title:", title)
        if t in token_tag_list:
            token_match = token_match + 1
            token_tag_list.remove(t)  # To avoid counting duplicates
        else:
            token_mismatch = token_mismatch + 1
    #print(tokens)
    #print(pred_tag)

print(f"Total Token Matches: {token_match}, Total Token Mismatches: {token_mismatch}")